<a href="https://colab.research.google.com/github/Natalyrubin/AI-Chat/blob/main/NatalyRubin_FinalProject_Bullying.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Upload Data | op-1 | import from kaggle

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os


path = kagglehub.dataset_download("leomartinelli/bullying-in-schools")
print(os.listdir(path))

df = pd.read_csv(os.path.join(path, "Bullying_2018.csv"))
df.head()

In [ ]:
df = pd.read_csv(os.path.join(path, "Bullying_2018.csv"), sep=';')

In [ ]:
df.head()

*
*
*
*

END | op-1
---


*
*
*
*

# Upload Data | op-2 | import from local

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("Bullying_2018.csv")

In [ ]:
df.head()


In [ ]:
df = pd.read_csv("Bullying_2018.csv", sep=';')

In [ ]:
df.head()

*
*
*
*

END | op-2
---


*
*
*
*



---


# Data Overview & Missing Values Analysis

In [ ]:
df.info()
df.describe()
df.sample()

In [ ]:
def count_empty_after_strip(col):
        return col.isna().sum() + col.astype(str).str.strip().eq('').sum()

empty_counts = df.apply(count_empty_after_strip)
print(empty_counts)

In [ ]:
(empty_counts / len(df)).sort_values() * 100

In [ ]:
df = df.apply(lambda col: col.mask(col.astype(str).str.strip() == '', np.nan))

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df[['Were_underweight', 'Were_overweight', 'Were_obese']].notna().all(axis=1).sum()

In [ ]:
df[['Were_underweight', 'Were_overweight', 'Were_obese']].isna().all(axis=1).sum()



---


# Data Cleaning & Preparation

בחינה האם כדאי לעבוד עם הדאטה המלא ולפלטר משקל בשאלות משקל / או לפלטר מראש ולעבוד רק עם מי שענה על שאלות משקל.
על מנת לא להטות את המחקר כתוצאה מההשערה שמי שלא ענה זה כתוצאה מבושה במשקל וככל הנראה משפיע על התוצאה.

In [ ]:
df_analysis = df.copy()

In [ ]:
df_analysis["bullied"] = (
    (df["Bullied_on_school_property_in_past_12_months"] == "Yes") |
    (df["Bullied_not_on_school_property_in_past_12_months"] == "Yes") |
    (df["Cyber_bullied_in_past_12_months"] == "Yes")
).astype(int)

In [ ]:
df_analysis

In [ ]:
df_analysis["bullied"].mean()

In [ ]:
df_analysis["weight_missing"] = df["Were_underweight"].isnull().astype(int)

In [ ]:
pd.crosstab(df_analysis["weight_missing"], df_analysis["bullied"], normalize="index")

לא נמצאה עדות להבדל משמעותי בשיעור הבריונות בין תלמידים שדיווחו על משקל לבין אלו שלא, ולכן נבחר לעבוד עם כלל המדגם. עם זאת, ניתוח משתני המשקל יבוצע על תת-אוכלוסייה בלבד



---


# Setup: Install & Import Libraries

In [ ]:
# System & DB connectivity (Optional / Future use)
#!apt-get install -y libsasl2-dev
#!pip install thrift
#!pip install sasl
#!pip install thrift_sasl
#!pip install sqlalchemy
#!pip install impyla
#!pip install pure-sasl


In [ ]:
# Visualization (Used in EDA)
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning - Classification (Used)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Machine Learning - General (Used)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Machine Learning - Regression (Optional / Future use)
#from sklearn.linear_model import LinearRegression

# Model Evaluation (Used)
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    roc_curve,
    auc
)

# Statistical Analysis (Used)
from scipy.stats import (
    chi2_contingency,
    norm,
    ttest_ind,
    ttest_1samp,
    t,
    pearsonr,
    spearmanr,
    binom
)

# Database Connectivity (Optional / Future use)
#import sqlalchemy
#from impala.dbapi import connect

# Tree Visualization (Optional)
#from sklearn import tree

In [ ]:
df_analysis["Felt_lonely_num"] = df_analysis["Felt_lonely"].map({
    "Never": 0,
    "Rarely": 1,
    "Sometimes": 2,
    "Most of the time": 3,
    "Always": 4
})

df_analysis["Close_friends_num"] = df_analysis["Close_friends"].map({
    "0": 0,
    "1": 1,
    "2": 2,
    "3 or more": 3
})

df_analysis["Sex_num"] = df_analysis["Sex"].map({
    "Male": 0,
    "Female": 1
})

In [ ]:
pd.crosstab(df_analysis["Felt_lonely_num"], df_analysis["bullied"], normalize="index") * 100

In [ ]:
lonely_table = pd.crosstab(df_analysis["Felt_lonely_num"], df_analysis["bullied"])
lonely_table

In [ ]:
lonely_table_pct = pd.crosstab(
    df_analysis["Felt_lonely"],
    df_analysis["bullied"],
    normalize="index"
)

order = ["Never", "Rarely", "Sometimes", "Most of the time", "Always"]
lonely_table_pct = lonely_table_pct.reindex(order)

lonely_table_pct.plot(kind="bar", figsize=(8,5))
plt.title("Bullying Rate by Loneliness Level")
plt.xlabel("Felt Lonely")
plt.ylabel("Percentage")
plt.legend(["Not Bullied", "Bullied"])
plt.show()

In [ ]:
lonely_table_pct[1].plot(marker='o', figsize=(8,5))
plt.title("Bullying Probability by Loneliness Level")
plt.xlabel("Felt Lonely")
plt.ylabel("Probability")
plt.grid()
plt.show()

In [ ]:
sns.heatmap(lonely_table_pct, annot=True, cmap="Reds", fmt=".2f")
plt.title("Bullying Rate Heatmap")
plt.xlabel("Bullied")
plt.ylabel("Felt Lonely")
plt.show()

In [ ]:
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(lonely_table)

lonely_expected_df = pd.DataFrame(expected,
                           index=lonely_table.index,
                           columns=lonely_table.columns)

lonely_expected_df

In [ ]:
print("Chi-Square Test Results:")
print("------------------------")
print(f"Chi2 Statistic: {chi2:.2f}")
print(f"P-value: {p:.5f}")
print(f"Degrees of Freedom: {dof}")

alpha = 0.05
if p < alpha:
    print("\nResult: Significant relationship (reject H0)")
else:
    print("\nResult: No significant relationship (fail to reject H0)")

In [ ]:
df_analysis["Felt_lonely"].value_counts(dropna=False)

In [ ]:
lonely_model_data = df_analysis[["Felt_lonely_num", "bullied"]].dropna()

X = lonely_model_data[["Felt_lonely_num"]]
y = lonely_model_data["bullied"]

model = LogisticRegression()
model.fit(X, y)

In [ ]:
model.coef_


In [ ]:
model.intercept_

In [ ]:
x_vals = np.array([0,1,2,3,4])
probs = model.predict_proba(x_vals.reshape(-1,1))[:,1]

plt.plot(x_vals, probs, marker='o')
plt.xticks(x_vals, ["Never","Rarely","Sometimes","Most","Always"])
plt.ylabel("Probability of Being Bullied")
plt.title("Logistic Regression Effect")
plt.grid()
plt.show()

In [ ]:
spearmanr(
    lonely_model_data["Felt_lonely_num"],
    lonely_model_data["bullied"]
)

In [ ]:
pd.crosstab(df_analysis["Close_friends_num"], df_analysis["bullied"], normalize="index") * 100

In [ ]:
friends_table = pd.crosstab(df_analysis["Close_friends_num"], df_analysis["bullied"])
friends_table

In [ ]:
friends_table_pct = pd.crosstab(
    df_analysis["Close_friends"],
    df_analysis["bullied"],
    normalize="index"
)

friends_order = ["0", "1", "2", "3 or more"]
friends_table_pct = friends_table_pct.reindex(friends_order)

friends_table_pct.plot(kind="bar", figsize=(8,5))
plt.title("Bullying Rate by Number of Close Friends")
plt.xlabel("Number of Close Friends")
plt.ylabel("Percentage")
plt.legend(["Not Bullied", "Bullied"])
plt.show()

In [ ]:
friends_table_pct[1].plot(marker='o', figsize=(8,5))
plt.title("Bullying Probability by Number of Close Friends")
plt.xlabel("Number of Close Friends")
plt.ylabel("Probability")
plt.grid()
plt.show()

In [ ]:
sns.heatmap(friends_table_pct, annot=True, cmap="Reds", fmt=".2f")
plt.title("Bullying Rate Heatmap - Close Friends")
plt.xlabel("Bullied")
plt.ylabel("Number of Close Friends")
plt.show()

In [ ]:
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(friends_table)

friends_expected_df = pd.DataFrame(
    expected,
    index=friends_table.index,
    columns=friends_table.columns
)

friends_expected_df

In [ ]:
print("Chi-Square Test Results:")
print("------------------------")
print(f"Chi2 Statistic: {chi2:.2f}")
print(f"P-value: {p:.5f}")
print(f"Degrees of Freedom: {dof}")

alpha = 0.05
if p < alpha:
    print("\nResult: Significant relationship (reject H0)")
else:
    print("\nResult: No significant relationship (fail to reject H0)")

In [ ]:
df_analysis["Close_friends"].value_counts(dropna=False)

In [ ]:
friends_model_data = df_analysis[["Close_friends_num", "bullied"]].dropna()

X = friends_model_data[["Close_friends_num"]]
y = friends_model_data["bullied"]

friends_model = LogisticRegression()
friends_model.fit(X, y)

friends_model.coef_

In [ ]:
friends_model.intercept_

In [ ]:
x_vals = np.array([0,1,2,3])
probs = friends_model.predict_proba(x_vals.reshape(-1,1))[:,1]

plt.plot(x_vals, probs, marker='o')
plt.xticks(x_vals, ["0","1","2","3+"])
plt.ylabel("Probability of Being Bullied")
plt.title("Logistic Regression Effect - Close Friends")
plt.grid()
plt.show()

In [ ]:
spearmanr(
    friends_model_data["Close_friends_num"],
    friends_model_data["bullied"]
)

In [ ]:
pd.crosstab(df_analysis["Sex_num"], df_analysis["bullied"], normalize="index") * 100

In [ ]:
sex_table = pd.crosstab(df_analysis["Sex_num"], df_analysis["bullied"])
sex_table

In [ ]:
sex_table_pct = pd.crosstab(
    df_analysis["Sex"],
    df_analysis["bullied"],
    normalize="index"
)

sex_table_pct

In [ ]:
sex_table_pct.plot(kind="bar", figsize=(6,4))
plt.title("Bullying Rate by Sex")
plt.xlabel("Sex")
plt.ylabel("Percentage")
plt.legend(["Not Bullied", "Bullied"])
plt.show()

In [ ]:
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(sex_table)

sex_expected_df = pd.DataFrame(
    expected,
    index=sex_table.index,
    columns=sex_table.columns
)

sex_expected_df

In [ ]:
print("Chi-Square Test Results:")
print("------------------------")
print(f"Chi2 Statistic: {chi2:.2f}")
print(f"P-value: {p:.5f}")
print(f"Degrees of Freedom: {dof}")

if p < 0.05:
    print("\nResult: Significant relationship (reject H0)")
else:
    print("\nResult: No significant relationship (fail to reject H0)")

In [ ]:
sex_model_data = df_analysis[["Sex_num", "bullied"]].dropna()

X = sex_model_data[["Sex_num"]]
y = sex_model_data["bullied"]

sex_model = LogisticRegression()
sex_model.fit(X, y)

sex_model.coef_

In [ ]:
sex_model.intercept_

In [ ]:
from statsmodels.stats.proportion import proportions_ztest


count = sex_table[1].values

nobs = sex_table.sum(axis=1).values

stat, pval = proportions_ztest(count, nobs)

print(stat, pval)

In [ ]:
odds = (sex_table[1] / sex_table[0])
odds_ratio = odds[1] / odds[0]
odds_ratio

In [ ]:
cols = [
    "Physically_attacked",
    "Physical_fighting",
    "Miss_school_no_permission",
    "Other_students_kind_and_helpful",
    "Parents_understand_problems",
    "Missed_classes_or_school_without_permission",
    "Custom_Age"
]

for col in cols:
    print(f"\n===== {col} =====")
    print(df_analysis[col].value_counts(dropna=False))